In [308]:
import json
import pandas as pd
from pathlib import Path
import numpy as np

In [309]:
# 1. Load the raw JSON (list of nested dicts)
data_path = Path("../data/raw_credit_applications.json")  # or the correct relative/path in your repo
with open(data_path, "r") as f:
    raw_data = json.load(f)

# Quick sanity check
len(raw_data), type(raw_data[0])

# 2. Normalize nested JSON into a flat table
df = pd.json_normalize(raw_data)

# 3. Inspect and save
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 502 entries, 0 to 501
Data columns (total 21 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   _id                               502 non-null    str    
 1   spending_behavior                 502 non-null    object 
 2   processing_timestamp              62 non-null     str    
 3   applicant_info.full_name          502 non-null    str    
 4   applicant_info.email              502 non-null    str    
 5   applicant_info.ssn                497 non-null    str    
 6   applicant_info.ip_address         497 non-null    str    
 7   applicant_info.gender             501 non-null    str    
 8   applicant_info.date_of_birth      501 non-null    str    
 9   applicant_info.zip_code           501 non-null    str    
 10  financials.annual_income          497 non-null    object 
 11  financials.credit_history_months  502 non-null    int64  
 12  financials.debt_to_

In [310]:
# 2. Check for duplicate application IDs
total_rows = df.shape[0]
unique_ids = df["_id"].nunique()
duplicate_rows = total_rows - unique_ids

print("Total rows:", total_rows)
print("Unique _id values:", unique_ids)
print("Duplicate rows (by _id):", duplicate_rows)

before = df.shape[0]
df = df.drop_duplicates(subset="_id", keep="first")
after = df.shape[0]

print("Rows before dropping duplicates:", before)
print("Rows after dropping duplicates:", after)
print("Rows removed:", before - after)

Total rows: 502
Unique _id values: 500
Duplicate rows (by _id): 2
Rows before dropping duplicates: 502
Rows after dropping duplicates: 500
Rows removed: 2


In [311]:
# combin salary and income into one column
# done before to drop salary in next step before cleaning data
print("Pre-merge: income NaN:", df['financials.annual_income'].isnull().sum())
df['financials.annual_income'] = df['financials.annual_income'].fillna(df['financials.annual_salary'])

print("Post-merge income NaN:", df['financials.annual_income'].isnull().sum())

Pre-merge: income NaN: 5
Post-merge income NaN: 0


In [312]:
# 1. Overview of missing data
def missing_summary(df):
    missing_counts = df.isna().sum().sort_values(ascending=False)
    missing_percent = (missing_counts / len(df) * 100).round(1)
    return pd.DataFrame({
        "missing_count": missing_counts,
        "missing_percent": missing_percent
    })

ms = missing_summary(df)
print(ms)

                                  missing_count  missing_percent
notes                                       500            100.0
financials.annual_salary                    495             99.0
loan_purpose                                450             90.0
processing_timestamp                        438             87.6
decision.rejection_reason                   292             58.4
decision.approved_amount                    208             41.6
decision.interest_rate                      208             41.6
applicant_info.ssn                            4              0.8
applicant_info.ip_address                     4              0.8
financials.debt_to_income                     0              0.0
decision.loan_approved                        0              0.0
financials.savings_balance                    0              0.0
_id                                           0              0.0
financials.credit_history_months              0              0.0
spending_behavior        

In [313]:
# Drop columns with >50% missing values
threshold = 50  # Use value, not fraction
cols_to_drop = ms[ms["missing_percent"] > threshold].index.tolist()
print("Dropping columns with >50% missing:", cols_to_drop)

df = df.drop(columns=cols_to_drop)
print("Remaining columns:", df.shape[1])

Dropping columns with >50% missing: ['notes', 'financials.annual_salary', 'loan_purpose', 'processing_timestamp', 'decision.rejection_reason']
Remaining columns: 16


In [314]:
# 4. Ensure numeric columns are numeric
numeric_cols = [
    "financials.annual_income",
    "financials.credit_history_months",
    "financials.debt_to_income",
    "financials.savings_balance",
    "decision.interest_rate",
    "decision.approved_amount",
]

for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

print(df[numeric_cols].dtypes)

financials.annual_income            float64
financials.credit_history_months      int64
financials.debt_to_income           float64
financials.savings_balance            int64
decision.interest_rate              float64
decision.approved_amount            float64
dtype: object


In [315]:
# handle Datime consistency and transforming to age
today = pd.to_datetime('2026-03-05')
df['applicant_info.date_of_birth'] = pd.to_datetime(df['applicant_info.date_of_birth'], dayfirst=False, yearfirst=False, errors='coerce')
df['applicant_info.age'] = (today - df['applicant_info.date_of_birth']).dt.days // 365

# check gender for consistency
df['applicant_info.gender'] = df['applicant_info.gender'].str.strip().str.upper()

df['applicant_info.gender'] = df['applicant_info.gender'].map({
    'M': 'Male',
    'MALE': 'Male',
    'F': 'Female', 
    'FEMALE': 'Female',
    '': np.nan
}).ffill()

# unify email adresses and check for consistency
df['applicant_info.email'] = df['applicant_info.email'].str.lower().str.strip()
invalid_email = ~df['applicant_info.email'].str.contains(r'@[^@]+\.[^@]+$', na=False)
print(f"Invalid emails (no @domain.tld): {invalid_email.sum()} ({invalid_email.mean()*100:.1f}%)")
df.loc[invalid_email, 'applicant_info.email'] = np.nan

# check names for consistency
single_word_names = df['applicant_info.full_name'].str.split().str.len() == 1
no_capital = ~df['applicant_info.full_name'].str[0].str.isupper()
print(f"Single-word names: {single_word_names.sum()} ({single_word_names.mean()*100:.1f}%)")
print(f"No capital start: {no_capital.sum()} ({no_capital.mean()*100:.1f}%)")

# transform ssn patterns
ssn_pattern = r'^\d{3}-\d{2}-\d{4}$'
invalid_ssn = ~df['applicant_info.ssn'].str.match(ssn_pattern, na=False)
print(f"Invalid SSN format (not XXX-XX-XXXX): {invalid_ssn.sum()} ({invalid_ssn.mean()*100:.1f}%)")
df.loc[invalid_ssn, 'applicant_info.ssn'] = np.nan

# ZIP: Enforce 5 digits (truncate extra)
zip_str = df['applicant_info.zip_code'].astype(str)
long_zip = zip_str.str.len() > 5
print(f"ZIP >5 digits: {long_zip.sum()}")
df.loc[long_zip, 'applicant_info.zip_code'] = zip_str.loc[long_zip].str[:5]
df['applicant_info.zip_code'] = pd.to_numeric(df['applicant_info.zip_code'], errors='coerce').astype('Int64')

# non-boolean loan_approved
print("Non-boolean count loan_approved:", (~df['decision.loan_approved'].isin([True, False])).sum())

# 5. Simple sanity checks on numeric values
print("Negative income:", df[df["financials.annual_income"] < 0].shape[0])
print("Very high income (> 1,000,000):", df[df["financials.annual_income"] > 1_000_000].shape[0])
print("Negative savings:", df[df["financials.savings_balance"] < 0].shape[0])
neg_savings = (df["financials.savings_balance"] < 0).sum()
print(f"Negative saving: {neg_savings.sum()} ({(neg_savings.sum() / len(df)) * 100:.1f}%)")
df.loc[df["financials.savings_balance"] < 0, "financials.savings_balance"] = np.nan
print("Weird debt_to_income (<0 or >5):", df[(df["financials.debt_to_income"] < 0) | (df["financials.debt_to_income"] > 5)].shape[0])

# Calculate absolute debt = income * debt_to_income
df['applicant_info.debt'] = df['financials.annual_income'] * df['financials.debt_to_income']

df

Invalid emails (no @domain.tld): 10 (2.0%)
Single-word names: 0 (0.0%)
No capital start: 0 (0.0%)
Invalid SSN format (not XXX-XX-XXXX): 4 (0.8%)
ZIP >5 digits: 0
Non-boolean count loan_approved: 0
Negative income: 0
Very high income (> 1,000,000): 0
Negative savings: 1
Negative saving: 1 (0.2%)
Weird debt_to_income (<0 or >5): 0


,_id,spending_behavior,applicant_info.full_name,applicant_info.email,applicant_info.ssn,applicant_info.ip_address,applicant_info.gender,applicant_info.date_of_birth,applicant_info.zip_code,financials.annual_income,financials.credit_history_months,financials.debt_to_income,financials.savings_balance,decision.loan_approved,decision.interest_rate,decision.approved_amount,applicant_info.age,applicant_info.debt
0,app_200,"[{'category': 'Shopping', 'amount': 480}, {'ca...",Jerry Smith,jerry.smith17@hotmail.com,596-64-4340,192.168.48.155,Male,2001-03-09,10036,73000.0,23,0.20,31212.0,False,NaN,NaN,25.0,14600.0
1,app_037,"[{'category': 'Rent', 'amount': 608}, {'catego...",Brandon Walker,brandon.walker2@yahoo.com,425-69-4784,10.1.102.112,Male,1992-03-31,10032,78000.0,51,0.18,17915.0,False,NaN,NaN,33.0,14040.0
2,app_215,"[{'category': 'Rent', 'amount': 109}]",Scott Moore,scott.moore94@mail.com,370-78-5178,10.240.193.250,Male,1989-10-24,10075,61000.0,41,0.21,37909.0,True,3.7,59000.0,36.0,12810.0
3,app_024,"[{'category': 'Fitness', 'amount': 575}]",Thomas Lee,thomas.lee6@protonmail.com,194-35-1833,192.168.175.67,Male,1983-04-25,10077,103000.0,70,0.35,0.0,True,4.3,34000.0,42.0,36050.0
4,app_184,"[{'category': 'Entertainment', 'amount': 463}]",Brian Rodriguez,brian.rodriguez86@aol.com,480-41-2475,172.29.125.105,Male,1999-05-21,10080,57000.0,14,0.23,31763.0,False,NaN,NaN,26.0,13110.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
497,app_468,"[{'category': 'Transportation', 'amount': 701}]",Patrick Martinez,patrick.martinez26@aol.com,996-86-6099,172.31.95.30,Male,1999-05-23,10046,22000.0,35,0.35,8982.0,False,NaN,NaN,26.0,7700.0
498,app_192,"[{'category': 'Healthcare', 'amount': 650}]",Dennis Lopez,dennis.lopez78@yahoo.com,936-44-8002,172.29.183.18,Male,NaT,10088,78000.0,40,0.22,34292.0,True,6.0,18000.0,NaN,17160.0
499,app_234,"[{'category': 'Insurance', 'amount': 526}]",Samuel Hill,samuel.hill67@protonmail.com,937-72-8731,10.143.146.157,Male,NaT,10090,96000.0,60,0.30,38703.0,False,NaN,NaN,NaN,28800.0
500,app_306,"[{'category': 'Insurance', 'amount': 490}]",Anna White,anna.white6@gmail.com,757-27-8131,10.26.176.56,Female,1978-07-26,90227,106000.0,80,0.29,63560.0,True,3.1,57000.0,47.0,30740.0


In [316]:
# Basic data-quality overview
print(missing_summary(df))

                                  missing_count  missing_percent
decision.approved_amount                    208             41.6
decision.interest_rate                      208             41.6
applicant_info.age                          161             32.2
applicant_info.date_of_birth                161             32.2
applicant_info.email                         10              2.0
applicant_info.ssn                            4              0.8
applicant_info.ip_address                     4              0.8
financials.savings_balance                    1              0.2
applicant_info.zip_code                       1              0.2
_id                                           0              0.0
decision.loan_approved                        0              0.0
financials.annual_income                      0              0.0
financials.debt_to_income                     0              0.0
financials.credit_history_months              0              0.0
spending_behavior        

In [317]:
path = Path("../data/clean_credit_applications.csv")
df.to_csv(path, index=False)
print("Shape:", df.shape)

Shape: (500, 18)
